# Clustering Countries for Tourism Services

## Objective
Analyze key statistical indicators to group countries and identify locations for tourism investment.

## Key Questions
- Which variables should be used for clustering?
- How many clusters can be identified?
- How do clusters vary?
- How to use PCA to retain 90% variance?
- How to perform clustering on PCA components?

## 1. Import Libraries and Load Dataset

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

In [ ]:
# Load dataset
df = pd.read_csv('country_stats.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Basic information
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing, 'Percentage': missing_pct})
missing_df[missing_df['Missing_Count'] > 0].sort_values('Percentage', ascending=False)

In [ ]:
# Visualize missing data
plt.figure(figsize=(14, 6))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title('Missing Data Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of key numeric features
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in ['country']]

fig, axes = plt.subplots(4, 3, figsize=(15, 12))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols[:12]):
    axes[idx].hist(df[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_title(col, fontsize=10)
    axes[idx].set_xlabel('')
    
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Create a copy for processing
df_clean = df.copy()

# Store country names for later use
countries = df_clean['country'].values

# Select relevant features for clustering (excluding identifiers)
features_to_exclude = ['country', 'Region']
clustering_features = [col for col in df_clean.columns if col not in features_to_exclude]

print(f"Features for clustering: {len(clustering_features)}")
print(clustering_features)

In [ ]:
# Handle missing values - drop rows with too many missing values
# Keep rows with at least 70% non-null values
threshold = 0.7 * len(clustering_features)
df_clean = df_clean.dropna(thresh=threshold)

print(f"Rows after dropping sparse data: {len(df_clean)}")

# Fill remaining missing values with median
for col in clustering_features:
    if df_clean[col].isnull().any():
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

In [ ]:
# Prepare data for clustering
X = df_clean[clustering_features].values
countries_clean = df_clean['country'].values

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Scaled data shape: {X_scaled.shape}")

## 4. Clustering Analysis (Original Features)

In [ ]:
# Determine optimal number of clusters using Elbow Method
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Plot Elbow curve and Silhouette scores
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(K_range, inertias, 'bo-')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method')
ax1.grid(True)

ax2.plot(K_range, silhouette_scores, 'ro-')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score vs Number of Clusters')
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Hierarchical clustering dendrogram
plt.figure(figsize=(15, 7))
linkage_matrix = linkage(X_scaled, method='ward')
dendrogram(linkage_matrix, truncate_mode='lastp', p=30)
plt.title('Hierarchical Clustering Dendrogram (Truncated)')
plt.xlabel('Cluster Size')
plt.ylabel('Distance')
plt.show()

In [ ]:
# Apply K-Means with optimal k (adjust based on elbow/silhouette)
optimal_k = 4  # Adjust this based on the plots above

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

# Add cluster labels to dataframe
df_clean['Cluster'] = clusters

print(f"\nCluster distribution:")
print(df_clean['Cluster'].value_counts().sort_index())

## 5. Cluster Profiling

In [ ]:
# Cluster statistics
cluster_profile = df_clean.groupby('Cluster')[clustering_features].mean()
cluster_profile

In [ ]:
# Sample countries from each cluster
for cluster_id in range(optimal_k):
    print(f"\n{'='*60}")
    print(f"Cluster {cluster_id} - Sample Countries:")
    print('='*60)
    cluster_countries = df_clean[df_clean['Cluster'] == cluster_id]['country'].head(10).tolist()
    print(', '.join(cluster_countries))

In [ ]:
# Visualize cluster characteristics using radar chart
from math import pi

# Select key features for visualization
key_features = ['GDP: Gross domestic product', 'Population in thousands', 
                'Mobile-cellular subscriptions', 'Individuals using the Internet',
                'Health: Total expenditure', 'Education: Government expenditure']

# Normalize cluster means for visualization
cluster_means_norm = cluster_profile[key_features].apply(lambda x: (x - x.min()) / (x.max() - x.min()))

# Create radar chart
angles = [n / len(key_features) * 2 * pi for n in range(len(key_features))]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

for cluster_id in range(optimal_k):
    values = cluster_means_norm.loc[cluster_id].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=f'Cluster {cluster_id}')
    ax.fill(angles, values, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([f.replace(': ', ':\n') for f in key_features], size=9)
ax.set_ylim(0, 1)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.set_title('Cluster Profiles - Key Features', size=14, pad=20)
plt.tight_layout()
plt.show()

## 6. PCA for Dimensionality Reduction (90% Variance)

In [ ]:
# Apply PCA to retain 90% variance
pca = PCA(n_components=0.90, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Original features: {X_scaled.shape[1]}")
print(f"PCA components (90% variance): {X_pca.shape[1]}")
print(f"\nExplained variance ratio: {pca.explained_variance_ratio_}")
print(f"Cumulative variance explained: {pca.explained_variance_ratio_.cumsum()[-1]:.4f}")

In [ ]:
# Visualize explained variance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Individual variance
ax1.bar(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_)
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance Ratio')
ax1.set_title('Variance Explained by Each Component')
ax1.grid(True, alpha=0.3)

# Cumulative variance
ax2.plot(range(1, len(pca.explained_variance_ratio_) + 1), 
         pca.explained_variance_ratio_.cumsum(), 'ro-')
ax2.axhline(y=0.90, color='g', linestyle='--', label='90% Variance')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Explained Variance')
ax2.set_title('Cumulative Variance Explained')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance in principal components
components_df = pd.DataFrame(
    pca.components_,
    columns=clustering_features,
    index=[f'PC{i+1}' for i in range(len(pca.components_))]
)

# Plot heatmap of component loadings
plt.figure(figsize=(14, 6))
sns.heatmap(components_df, cmap='RdBu_r', center=0, annot=False, cbar=True)
plt.title('PCA Component Loadings')
plt.xlabel('Original Features')
plt.ylabel('Principal Components')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Clustering on PCA Components

In [ ]:
# Determine optimal clusters for PCA data
inertias_pca = []
silhouette_scores_pca = []

for k in K_range:
    kmeans_pca = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_pca.fit(X_pca)
    inertias_pca.append(kmeans_pca.inertia_)
    silhouette_scores_pca.append(silhouette_score(X_pca, kmeans_pca.labels_))

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(K_range, inertias_pca, 'bo-')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method (PCA Data)')
ax1.grid(True)

ax2.plot(K_range, silhouette_scores_pca, 'ro-')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score (PCA Data)')
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Apply K-Means on PCA data
optimal_k_pca = 4  # Adjust based on plots

kmeans_pca = KMeans(n_clusters=optimal_k_pca, random_state=42, n_init=10)
clusters_pca = kmeans_pca.fit_predict(X_pca)

df_clean['Cluster_PCA'] = clusters_pca

print(f"\nPCA Cluster distribution:")
print(df_clean['Cluster_PCA'].value_counts().sort_index())

In [ ]:
# Visualize clusters in 2D PCA space
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters_pca, 
                     cmap='viridis', s=100, alpha=0.6, edgecolors='black')
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.title('Countries Clustered in PCA Space')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compare original vs PCA clustering
comparison = pd.crosstab(df_clean['Cluster'], df_clean['Cluster_PCA'], 
                         rownames=['Original'], colnames=['PCA'])
print("\nCluster Comparison (Original vs PCA):")
print(comparison)

## 8. Cluster Profiling (PCA Clusters)

In [ ]:
# Profile PCA clusters
for cluster_id in range(optimal_k_pca):
    print(f"\n{'='*70}")
    print(f"PCA Cluster {cluster_id}")
    print('='*70)
    
    cluster_data = df_clean[df_clean['Cluster_PCA'] == cluster_id]
    print(f"Number of countries: {len(cluster_data)}")
    
    print(f"\nSample countries:")
    print(', '.join(cluster_data['country'].head(15).tolist()))
    
    print(f"\nKey statistics:")
    key_stats = cluster_data[['GDP: Gross domestic product', 'Population in thousands',
                              'Mobile-cellular subscriptions', 'Individuals using the Internet']].mean()
    print(key_stats)

## 9. Business Recommendations and Conclusions

In [ ]:
# Create summary visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# GDP by cluster
df_clean.groupby('Cluster_PCA')['GDP: Gross domestic product'].mean().plot(kind='bar', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Average GDP by Cluster')
axes[0,0].set_xlabel('Cluster')
axes[0,0].set_ylabel('GDP (Million USD)')

# Internet usage by cluster
df_clean.groupby('Cluster_PCA')['Individuals using the Internet'].mean().plot(kind='bar', ax=axes[0,1], color='coral')
axes[0,1].set_title('Average Internet Users by Cluster')
axes[0,1].set_xlabel('Cluster')
axes[0,1].set_ylabel('Users per 100 inhabitants')

# Mobile subscriptions by cluster
df_clean.groupby('Cluster_PCA')['Mobile-cellular subscriptions'].mean().plot(kind='bar', ax=axes[1,0], color='green')
axes[1,0].set_title('Average Mobile Subscriptions by Cluster')
axes[1,0].set_xlabel('Cluster')
axes[1,0].set_ylabel('Subscriptions per 100 inhabitants')

# Health expenditure by cluster
df_clean.groupby('Cluster_PCA')['Health: Total expenditure'].mean().plot(kind='bar', ax=axes[1,1], color='purple')
axes[1,1].set_title('Average Health Expenditure by Cluster')
axes[1,1].set_xlabel('Cluster')
axes[1,1].set_ylabel('% of GDP')

plt.tight_layout()
plt.show()

## Key Findings and Recommendations

### Analysis Summary:
1. **Number of Clusters**: Identified distinct groups of countries based on economic, social, and infrastructure indicators
2. **PCA Results**: Successfully reduced dimensionality while retaining 90% of variance
3. **Cluster Characteristics**: Each cluster represents countries with similar tourism potential profiles

### Business Recommendations:

**High-Priority Investment Clusters:**
- Countries with high GDP, good infrastructure (internet, mobile), and moderate tourism development
- These represent markets with purchasing power and connectivity but room for tourism growth

**Emerging Market Clusters:**
- Countries showing improving economic indicators and infrastructure
- Early investment can establish market presence before competition intensifies

**Established Market Clusters:**
- High GDP, excellent infrastructure, high internet/mobile penetration
- Focus on premium services and differentiation

**Developing Market Clusters:**
- Lower current indicators but potential for growth
- Consider long-term strategic partnerships

### Next Steps:
1. Deep-dive analysis of top clusters for investment
2. Market research in specific countries within priority clusters
3. Develop tailored tourism service packages for each cluster
4. Monitor cluster migration as countries develop

In [ ]:
# Export clustered data for further analysis
df_clean[['country', 'Region', 'Cluster', 'Cluster_PCA']].to_csv('country_clusters.csv', index=False)
print("Cluster assignments exported to 'country_clusters.csv'")